<a href="https://colab.research.google.com/github/seethaladevi2024-cpu/Edufyi-Tech-Project---Flask-DDoS-Detection/blob/main/Flask_DDoS_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from flask import Flask, request, render_template, redirect, url_for, flash, send_file, session
from sklearn.metrics import accuracy_score, classification_report
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.secret_key = 'ddos_detection_secret_key_2024'

UPLOAD_FOLDER = 'uploads'
STATIC_FOLDER = 'static'
ALLOWED_EXTENSIONS = {'csv'}

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 50 * 1024 * 1024  # 50MB max

os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(STATIC_FOLDER, exist_ok=True)


def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


def load_model():
    try:
        with open('model.pkl', 'rb') as f:
            model = pickle.load(f)
        return model
    except FileNotFoundError:
        return None


def preprocess_data(df):
    """Preprocess uploaded CSV data for prediction."""
    # Drop label column if present (for evaluation)
    label_col = None
    possible_label_cols = ['Label', 'label', 'LABEL', 'class', 'Class', 'target', 'Target']
    for col in possible_label_cols:
        if col in df.columns:
            label_col = col
            break

    true_labels = None
    if label_col:
        true_labels = df[label_col].copy()
        df = df.drop(columns=[label_col])

    # Drop non-numeric or identifier columns
    drop_cols = [c for c in df.columns if df[c].dtype == object]
    df = df.drop(columns=drop_cols, errors='ignore')

    # Handle inf / nan
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.median(numeric_only=True), inplace=True)

    return df, true_labels


def generate_charts(predictions):
    """Generate pie and bar charts from predictions."""
    unique, counts = np.unique(predictions, return_counts=True)
    count_dict = dict(zip(unique, counts))

    ddos_count = count_dict.get('DDoS', 0)
    benign_count = count_dict.get('BENIGN', 0)

    labels = []
    sizes = []
    colors_pie = []
    if ddos_count > 0:
        labels.append('DDoS')
        sizes.append(ddos_count)
        colors_pie.append('#e74c3c')
    if benign_count > 0:
        labels.append('BENIGN')
        sizes.append(benign_count)
        colors_pie.append('#2ecc71')

    # --- Pie Chart ---
    fig1, ax1 = plt.subplots(figsize=(6, 5))
    fig1.patch.set_facecolor('#f8f9fa')
    ax1.set_facecolor('#f8f9fa')
    wedges, texts, autotexts = ax1.pie(
        sizes, labels=labels, colors=colors_pie,
        autopct='%1.1f%%', startangle=140,
        textprops={'fontsize': 13, 'fontweight': 'bold'},
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    for at in autotexts:
        at.set_fontsize(12)
        at.set_color('white')
        at.set_fontweight('bold')
    ax1.set_title('Prediction Results', fontsize=16, fontweight='bold', pad=15, color='#2c3e50')
    plt.tight_layout()
    plt.savefig(os.path.join(STATIC_FOLDER, 'prediction_pie_chart.png'), dpi=120, bbox_inches='tight')
    plt.close(fig1)

    # --- Bar Chart ---
    fig2, ax2 = plt.subplots(figsize=(6, 5))
    fig2.patch.set_facecolor('#f8f9fa')
    ax2.set_facecolor('#f8f9fa')
    bar_labels = ['DDoS', 'BENIGN']
    bar_values = [ddos_count, benign_count]
    bar_colors = ['#3498db', '#f39c12']
    bars = ax2.bar(bar_labels, bar_values, color=bar_colors, edgecolor='white',
                   linewidth=1.5, width=0.5)
    for bar, val in zip(bars, bar_values):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(bar_values) * 0.01,
                 str(val), ha='center', va='bottom', fontsize=13, fontweight='bold', color='#2c3e50')
    ax2.set_title('Prediction Counts', fontsize=16, fontweight='bold', pad=15, color='#2c3e50')
    ax2.set_xlabel('Prediction', fontsize=13, labelpad=8, color='#555')
    ax2.set_ylabel('Count', fontsize=13, labelpad=8, color='#555')
    ax2.yaxis.grid(True, linestyle='--', alpha=0.6)
    ax2.set_axisbelow(True)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(STATIC_FOLDER, 'prediction_bar_chart.png'), dpi=120, bbox_inches='tight')
    plt.close(fig2)


@app.route('/', methods=['GET'])
def index():
    return render_template('index.html')


@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        flash('No file part in the request.', 'error')
        return redirect(url_for('index'))

    file = request.files['file']

    if file.filename == '':
        flash('No file selected. Please choose a CSV file.', 'error')
        return redirect(url_for('index'))

    if not allowed_file(file.filename):
        flash('Invalid file type. Please upload a CSV file.', 'error')
        return redirect(url_for('index'))

    filename = secure_filename(file.filename)
    filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
    file.save(filepath)

    # Load model
    model = load_model()
    if model is None:
        flash('Model file (model.pkl) not found. Please ensure the trained model is available.', 'error')
        return redirect(url_for('index'))

    # Load and preprocess CSV
    try:
        df_raw = pd.read_csv(filepath)
    except Exception as e:
        flash(f'Error reading CSV file: {str(e)}', 'error')
        return redirect(url_for('index'))

    df_features, true_labels = preprocess_data(df_raw.copy())

    if df_features.empty or len(df_features.columns) == 0:
        flash('No valid feature columns found in the uploaded CSV.', 'error')
        return redirect(url_for('index'))

    # Predict
    try:
        predictions = model.predict(df_features)
    except Exception as e:
        flash(f'Prediction error: {str(e)}', 'error')
        return redirect(url_for('index'))

    # Map predictions to labels if numeric
    label_map = {0: 'BENIGN', 1: 'DDoS'}
    if predictions.dtype != object:
        predictions = np.array([label_map.get(int(p), str(p)) for p in predictions])

    # Accuracy & Report
    accuracy = None
    report_str = None
    if true_labels is not None:
        true_mapped = true_labels.map(lambda x: 'DDoS' if str(x).upper() == 'DDOS' else ('BENIGN' if str(x).upper() == 'BENIGN' else str(x)))
        try:
            accuracy = accuracy_score(true_mapped, predictions) * 100
            report_str = classification_report(true_mapped, predictions, target_names=['BENIGN', 'DDoS'])
        except Exception:
            accuracy = None
            report_str = None

    if accuracy is None:
        # No ground truth — show distribution stats
        unique, counts = np.unique(predictions, return_counts=True)
        total = len(predictions)
        report_str = "No ground truth labels found in CSV.\n\nPrediction Distribution:\n"
        for u, c in zip(unique, counts):
            report_str += f"  {u}: {c} ({c/total*100:.1f}%)\n"
        accuracy = (np.sum(predictions == 'BENIGN') / len(predictions)) * 100  # % benign as proxy

    # Save prediction CSVs
    df_raw['Prediction'] = predictions
    ddos_df = df_raw[df_raw['Prediction'] == 'DDoS']
    benign_df = df_raw[df_raw['Prediction'] == 'BENIGN']
    ddos_df.to_csv('ddos_predictions.csv', index=False)
    benign_df.to_csv('benign_predictions.csv', index=False)

    # Generate charts
    generate_charts(predictions)

    flash('File processed successfully!', 'success')

    return render_template(
        'result.html',
        accuracy=f"{accuracy:.2f}",
        report=report_str,
        pie_chart='static/prediction_pie_chart.png',
        bar_chart='static/prediction_bar_chart.png',
        ddos_count=int(np.sum(predictions == 'DDoS')),
        benign_count=int(np.sum(predictions == 'BENIGN')),
        total_count=len(predictions)
    )


@app.route('/download/<file_type>')
def download(file_type):
    if file_type == 'ddos':
        path = 'ddos_predictions.csv'
        name = 'ddos_predictions.csv'
    elif file_type == 'benign':
        path = 'benign_predictions.csv'
        name = 'benign_predictions.csv'
    else:
        flash('Invalid download request.', 'error')
        return redirect(url_for('index'))

    if not os.path.exists(path):
        flash('File not found. Please run a prediction first.', 'error')
        return redirect(url_for('index'))

    return send_file(path, as_attachment=True, download_name=name)


if __name__ == '__main__':
    app.run(debug=True, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>DDoS Detection System</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/css/bootstrap.min.css" rel="stylesheet"/>
  <link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet"/>
  <link href="{{ url_for('static', filename='style.css') }}" rel="stylesheet"/>
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.3/font/bootstrap-icons.min.css"/>
</head>
<body>

  <!-- Header -->
  <header class="site-header">
    <div class="header-inner container">
      <div class="header-icon"><i class="bi bi-shield-shaded"></i></div>
      <h1 class="header-title">DDoS Detection System</h1>
      <p class="header-subtitle">Machine Learning — Powered Network Threat Analysis</p>
    </div>
    <div class="header-wave">
      <svg viewBox="0 0 1440 60" preserveAspectRatio="none" xmlns="http://www.w3.org/2000/svg">
        <path d="M0,30 C360,60 1080,0 1440,30 L1440,60 L0,60 Z" fill="#f0f4fa"/>
      </svg>
    </div>
  </header>

  <!-- Flash Messages -->
  <div class="container mt-3" style="max-width:680px;">
    {% with messages = get_flashed_messages(with_categories=true) %}
      {% if messages %}
        {% for category, message in messages %}
          <div class="alert alert-{{ 'danger' if category == 'error' else 'success' }} alert-dismissible fade show flash-alert" role="alert">
            <i class="bi bi-{{ 'exclamation-triangle-fill' if category == 'error' else 'check-circle-fill' }} me-2"></i>
            {{ message }}
            <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
          </div>
        {% endfor %}
      {% endif %}
    {% endwith %}
  </div>

  <!-- Main Upload Card -->
  <main class="container main-container">
    <div class="upload-card">
      <div class="card-body-inner">

        <div class="section-badge mb-3">
          <i class="bi bi-upload me-1"></i> Upload Dataset
        </div>

        <h2 class="upload-heading">Analyze Your Network Traffic</h2>
        <p class="upload-subtext">
          Upload a CSV file containing network flow features. The model will classify each record
          as <strong>DDoS</strong> or <strong>BENIGN</strong> and generate a full report.
        </p>

        <form method="POST" action="{{ url_for('predict') }}" enctype="multipart/form-data" id="uploadForm">
          <div class="file-drop-zone" id="dropZone">
            <div class="drop-icon"><i class="bi bi-file-earmark-spreadsheet"></i></div>
            <p class="drop-label">Drag &amp; drop your CSV here</p>
            <p class="drop-sub">or click to browse</p>
            <input type="file" name="file" id="fileInput" accept=".csv" class="file-hidden-input"/>
            <div id="fileNameDisplay" class="file-name-display d-none"></div>
          </div>

          <div class="mt-4 text-center">
            <button type="submit" class="btn-upload" id="submitBtn">
              <span class="btn-icon"><i class="bi bi-cpu-fill"></i></span>
              Upload and Predict
            </button>
          </div>
        </form>

        <!-- Info pills -->
        <div class="info-pills mt-4">
          <span class="info-pill"><i class="bi bi-filetype-csv me-1"></i>CSV Format</span>
          <span class="info-pill"><i class="bi bi-lightning-charge me-1"></i>Random Forest Model</span>
          <span class="info-pill"><i class="bi bi-bar-chart-line me-1"></i>Instant Analytics</span>
        </div>
      </div>
    </div>

    <!-- Feature cards row -->
    <div class="row features-row g-3 mt-2">
      <div class="col-md-4">
        <div class="feature-card">
          <div class="feature-icon red"><i class="bi bi-shield-x"></i></div>
          <h5>DDoS Detection</h5>
          <p>Identifies malicious distributed denial-of-service traffic patterns.</p>
        </div>
      </div>
      <div class="col-md-4">
        <div class="feature-card">
          <div class="feature-icon green"><i class="bi bi-shield-check"></i></div>
          <h5>Benign Traffic</h5>
          <p>Classifies legitimate network flows with high precision.</p>
        </div>
      </div>
      <div class="col-md-4">
        <div class="feature-card">
          <div class="feature-icon blue"><i class="bi bi-graph-up-arrow"></i></div>
          <h5>Full Report</h5>
          <p>Provides accuracy score, classification report and visual charts.</p>
        </div>
      </div>
    </div>
  </main>

  <footer class="site-footer-bottom">
    <p>&copy; 2024 DDoS Detection System &mdash; ML Security Analytics</p>
  </footer>

  <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/js/bootstrap.bundle.min.js"></script>
  <script>
    const dropZone = document.getElementById('dropZone');
    const fileInput = document.getElementById('fileInput');
    const fileNameDisplay = document.getElementById('fileNameDisplay');
    const submitBtn = document.getElementById('submitBtn');

    dropZone.addEventListener('click', () => fileInput.click());

    dropZone.addEventListener('dragover', e => {
      e.preventDefault();
      dropZone.classList.add('dragover');
    });
    dropZone.addEventListener('dragleave', () => dropZone.classList.remove('dragover'));
    dropZone.addEventListener('drop', e => {
      e.preventDefault();
      dropZone.classList.remove('dragover');
      const file = e.dataTransfer.files[0];
      if (file && file.name.endsWith('.csv')) {
        const dt = new DataTransfer();
        dt.items.add(file);
        fileInput.files = dt.files;
        showFileName(file.name);
      }
    });

    fileInput.addEventListener('change', () => {
      if (fileInput.files[0]) showFileName(fileInput.files[0].name);
    });

    function showFileName(name) {
      fileNameDisplay.textContent = '📄 ' + name;
      fileNameDisplay.classList.remove('d-none');
      dropZone.classList.add('file-selected');
    }

    document.getElementById('uploadForm').addEventListener('submit', () => {
      submitBtn.innerHTML = '<span class="spinner-border spinner-border-sm me-2"></span>Processing…';
      submitBtn.disabled = true;
    });
  </script>
</body>
</html>

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>DDoS Detection — Results</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/css/bootstrap.min.css" rel="stylesheet"/>
  <link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet"/>
  <link href="{{ url_for('static', filename='style.css') }}" rel="stylesheet"/>
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.3/font/bootstrap-icons.min.css"/>
</head>
<body class="results-page">

  <!-- Header -->
  <header class="site-header">
    <div class="header-inner container">
      <div class="header-icon"><i class="bi bi-shield-shaded"></i></div>
      <h1 class="header-title">DDoS Detection System</h1>
      <p class="header-subtitle">Analysis Results Dashboard</p>
    </div>
    <div class="header-wave">
      <svg viewBox="0 0 1440 60" preserveAspectRatio="none" xmlns="http://www.w3.org/2000/svg">
        <path d="M0,30 C360,60 1080,0 1440,30 L1440,60 L0,60 Z" fill="#f0f4fa"/>
      </svg>
    </div>
  </header>

  <!-- Flash Messages -->
  <div class="container mt-3" style="max-width:900px;">
    {% with messages = get_flashed_messages(with_categories=true) %}
      {% if messages %}
        {% for category, message in messages %}
          <div class="alert alert-{{ 'danger' if category == 'error' else 'success' }} alert-dismissible fade show flash-alert" role="alert">
            <i class="bi bi-{{ 'exclamation-triangle-fill' if category == 'error' else 'check-circle-fill' }} me-2"></i>
            {{ message }}
            <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
          </div>
        {% endfor %}
      {% endif %}
    {% endwith %}
  </div>

  <main class="container results-container">

    <!-- ── Accuracy Banner ── -->
    <div class="accuracy-banner">
      <div class="accuracy-label">PREDICTION ACCURACY</div>
      <div class="accuracy-value">{{ accuracy }}%</div>
      <div class="accuracy-sub">Based on {{ total_count }} records analyzed</div>
      <div class="accuracy-pills">
        <span class="pill-red"><i class="bi bi-exclamation-octagon-fill me-1"></i>DDoS: {{ ddos_count }}</span>
        <span class="pill-green"><i class="bi bi-check-circle-fill me-1"></i>BENIGN: {{ benign_count }}</span>
      </div>
    </div>

    <!-- ── Classification Report ── -->
    <div class="result-card mt-4">
      <div class="result-card-header">
        <i class="bi bi-file-text-fill me-2"></i>CLASSIFICATION REPORT
      </div>
      <div class="result-card-body">
        <pre class="report-box">{{ report }}</pre>
      </div>
    </div>

    <!-- ── Charts Side by Side ── -->
    <div class="row g-4 mt-2">
      <div class="col-md-6">
        <div class="result-card chart-card">
          <div class="result-card-header">
            <i class="bi bi-pie-chart-fill me-2"></i>Prediction Pie Chart
          </div>
          <div class="result-card-body text-center">
            <img src="{{ url_for('static', filename='prediction_pie_chart.png') }}"
                 alt="Prediction Pie Chart"
                 class="chart-img"
                 onerror="this.src='https://via.placeholder.com/400x300?text=Chart+Not+Found'"/>
          </div>
        </div>
      </div>
      <div class="col-md-6">
        <div class="result-card chart-card">
          <div class="result-card-header">
            <i class="bi bi-bar-chart-fill me-2"></i>Prediction Bar Chart
          </div>
          <div class="result-card-body text-center">
            <img src="{{ url_for('static', filename='prediction_bar_chart.png') }}"
                 alt="Prediction Bar Chart"
                 class="chart-img"
                 onerror="this.src='https://via.placeholder.com/400x300?text=Chart+Not+Found'"/>
          </div>
        </div>
      </div>
    </div>

    <!-- ── Download Section ── -->
    <div class="result-card mt-4 download-card">
      <div class="result-card-header">
        <i class="bi bi-cloud-download-fill me-2"></i>DOWNLOAD CSV FILES
      </div>
      <div class="result-card-body">
        <p class="download-desc">Download the filtered prediction results as CSV files for further analysis.</p>
        <div class="download-btn-row">
          <a href="{{ url_for('download', file_type='ddos') }}" class="btn-download btn-download-red">
            <i class="bi bi-file-earmark-arrow-down-fill me-2"></i>
            Download DDoS Predictions CSV
            <span class="btn-count">{{ ddos_count }} records</span>
          </a>
          <a href="{{ url_for('download', file_type='benign') }}" class="btn-download btn-download-green">
            <i class="bi bi-file-earmark-arrow-down-fill me-2"></i>
            Download BENIGN Predictions CSV
            <span class="btn-count">{{ benign_count }} records</span>
          </a>
        </div>
      </div>
    </div>

    <!-- ── Upload Another ── -->
    <div class="text-center mt-4 mb-5">
      <a href="{{ url_for('index') }}" class="btn-upload-another">
        <i class="bi bi-arrow-repeat me-2"></i>Upload Another File
      </a>
    </div>

  </main>

  <footer class="site-footer-bottom">
    <p>&copy; 2024 DDoS Detection System &mdash; ML Security Analytics</p>
  </footer>

  <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/js/bootstrap.bundle.min.js"></script>
  <script>
    // Animate accuracy counter
    const el = document.querySelector('.accuracy-value');
    if (el) {
      const target = parseFloat(el.textContent);
      let current = 0;
      const step = target / 60;
      const timer = setInterval(() => {
        current = Math.min(current + step, target);
        el.textContent = current.toFixed(2) + '%';
        if (current >= target) clearInterval(timer);
      }, 16);
    }
  </script>
</body>
</html>

In [ ]:
/* =========================================
   DDoS Detection System — style.css
   ========================================= */

:root {
  --blue-dark:   #1a3a6b;
  --blue-mid:    #1e5cb3;
  --blue-light:  #3b82f6;
  --blue-glow:   #60a5fa;
  --accent-red:  #e74c3c;
  --accent-green:#16a34a;
  --bg:          #f0f4fa;
  --card-bg:     #ffffff;
  --text-dark:   #1e293b;
  --text-mid:    #475569;
  --text-muted:  #94a3b8;
  --shadow-sm:   0 2px 8px rgba(30,92,179,.10);
  --shadow-md:   0 6px 28px rgba(30,92,179,.14);
  --shadow-lg:   0 16px 56px rgba(30,92,179,.18);
  --radius:      16px;
}

* { box-sizing: border-box; margin: 0; padding: 0; }

body {
  font-family: 'DM Sans', sans-serif;
  background: var(--bg);
  color: var(--text-dark);
  min-height: 100vh;
}

/* ── Header ─────────────────────────────── */
.site-header {
  background: linear-gradient(135deg, var(--blue-dark) 0%, var(--blue-mid) 60%, var(--blue-light) 100%);
  padding: 48px 0 0;
  text-align: center;
  position: relative;
  overflow: hidden;
}
.site-header::before {
  content: '';
  position: absolute;
  inset: 0;
  background: radial-gradient(ellipse at 70% 40%, rgba(96,165,250,.18) 0%, transparent 60%);
  pointer-events: none;
}
.header-inner { position: relative; z-index: 1; }
.header-icon {
  font-size: 3rem;
  color: rgba(255,255,255,.85);
  margin-bottom: 10px;
  filter: drop-shadow(0 0 12px rgba(96,165,250,.5));
}
.header-title {
  font-family: 'Syne', sans-serif;
  font-size: clamp(1.8rem, 4vw, 2.8rem);
  font-weight: 800;
  color: #fff;
  letter-spacing: -.5px;
  text-shadow: 0 2px 16px rgba(0,0,0,.25);
}
.header-subtitle {
  color: rgba(255,255,255,.75);
  font-size: 1rem;
  margin-top: 6px;
  font-weight: 300;
  letter-spacing: .5px;
  padding-bottom: 28px;
}
.header-wave { line-height: 0; }
.header-wave svg { width: 100%; height: 60px; display: block; }

/* ── Flash Alerts ────────────────────────── */
.flash-alert {
  border-radius: 12px;
  border: none;
  font-weight: 500;
  box-shadow: var(--shadow-sm);
}

/* ── Main Container ──────────────────────── */
.main-container {
  max-width: 720px;
  padding-top: 36px;
  padding-bottom: 60px;
}

/* ── Upload Card ─────────────────────────── */
.upload-card {
  background: var(--card-bg);
  border-radius: var(--radius);
  box-shadow: var(--shadow-md);
  overflow: hidden;
}
.card-body-inner { padding: 40px 40px 32px; }

.section-badge {
  display: inline-flex;
  align-items: center;
  background: #e0eaff;
  color: var(--blue-mid);
  font-size: .78rem;
  font-weight: 600;
  letter-spacing: .8px;
  text-transform: uppercase;
  padding: 4px 12px;
  border-radius: 100px;
}

.upload-heading {
  font-family: 'Syne', sans-serif;
  font-size: 1.65rem;
  font-weight: 700;
  color: var(--text-dark);
  margin-top: 10px;
}
.upload-subtext {
  color: var(--text-mid);
  font-size: .97rem;
  line-height: 1.6;
  margin-top: 8px;
  margin-bottom: 24px;
}

/* Drop Zone */
.file-drop-zone {
  border: 2.5px dashed #c7d7f0;
  border-radius: 14px;
  padding: 40px 24px;
  text-align: center;
  cursor: pointer;
  background: #f7f9ff;
  transition: border-color .2s, background .2s, transform .15s;
  position: relative;
}
.file-drop-zone:hover,
.file-drop-zone.dragover {
  border-color: var(--blue-light);
  background: #edf2ff;
  transform: scale(1.01);
}
.file-drop-zone.file-selected {
  border-color: var(--accent-green);
  background: #f0fdf4;
}
.drop-icon {
  font-size: 2.8rem;
  color: var(--blue-light);
  margin-bottom: 10px;
}
.drop-label {
  font-size: 1rem;
  font-weight: 600;
  color: var(--text-dark);
  margin-bottom: 2px;
}
.drop-sub { font-size: .87rem; color: var(--text-muted); }
.file-hidden-input {
  position: absolute;
  inset: 0;
  opacity: 0;
  cursor: pointer;
  width: 100%;
  height: 100%;
}
.file-name-display {
  margin-top: 12px;
  font-size: .9rem;
  color: var(--accent-green);
  font-weight: 600;
}

/* Upload Button */
.btn-upload {
  display: inline-flex;
  align-items: center;
  gap: 10px;
  background: linear-gradient(135deg, var(--blue-dark) 0%, var(--blue-mid) 55%, var(--blue-light) 100%);
  color: #fff;
  font-family: 'Syne', sans-serif;
  font-size: 1.08rem;
  font-weight: 700;
  padding: 15px 44px;
  border-radius: 50px;
  border: none;
  cursor: pointer;
  box-shadow: 0 6px 24px rgba(30,92,179,.35);
  transition: transform .15s, box-shadow .15s, opacity .15s;
  letter-spacing: .3px;
}
.btn-upload:hover { transform: translateY(-2px); box-shadow: 0 10px 32px rgba(30,92,179,.45); }
.btn-upload:active { transform: translateY(0); }
.btn-upload:disabled { opacity: .7; cursor: not-allowed; }
.btn-icon { font-size: 1.2rem; }

/* Info Pills */
.info-pills { display: flex; flex-wrap: wrap; gap: 8px; justify-content: center; }
.info-pill {
  background: #eef2ff;
  color: var(--blue-mid);
  font-size: .8rem;
  font-weight: 500;
  padding: 5px 14px;
  border-radius: 100px;
  border: 1px solid #c7d7f0;
}

/* Feature Cards */
.features-row {}
.feature-card {
  background: var(--card-bg);
  border-radius: 14px;
  padding: 24px 20px;
  text-align: center;
  box-shadow: var(--shadow-sm);
  transition: transform .2s, box-shadow .2s;
  height: 100%;
}
.feature-card:hover { transform: translateY(-4px); box-shadow: var(--shadow-md); }
.feature-icon {
  width: 52px; height: 52px;
  border-radius: 14px;
  display: flex; align-items: center; justify-content: center;
  font-size: 1.5rem;
  margin: 0 auto 14px;
}
.feature-icon.red   { background: #fee2e2; color: var(--accent-red); }
.feature-icon.green { background: #dcfce7; color: var(--accent-green); }
.feature-icon.blue  { background: #dbeafe; color: var(--blue-mid); }
.feature-card h5 {
  font-family: 'Syne', sans-serif;
  font-size: 1rem;
  font-weight: 700;
  margin-bottom: 8px;
  color: var(--text-dark);
}
.feature-card p { font-size: .88rem; color: var(--text-mid); line-height: 1.5; }

/* ── Results Page ─────────────────────────── */
.results-container { max-width: 960px; padding-top: 36px; padding-bottom: 60px; }

/* Accuracy Banner */
.accuracy-banner {
  background: linear-gradient(135deg, var(--blue-dark) 0%, var(--blue-mid) 55%, var(--blue-light) 100%);
  border-radius: var(--radius);
  padding: 36px 40px;
  text-align: center;
  color: #fff;
  box-shadow: var(--shadow-lg);
  position: relative;
  overflow: hidden;
}
.accuracy-banner::before {
  content: '';
  position: absolute;
  inset: 0;
  background: radial-gradient(ellipse at 80% 50%, rgba(255,255,255,.08) 0%, transparent 60%);
}
.accuracy-label {
  font-size: .78rem;
  font-weight: 700;
  letter-spacing: 2.5px;
  text-transform: uppercase;
  opacity: .8;
  margin-bottom: 4px;
}
.accuracy-value {
  font-family: 'Syne', sans-serif;
  font-size: clamp(3rem, 8vw, 5.5rem);
  font-weight: 800;
  line-height: 1.1;
  letter-spacing: -2px;
  text-shadow: 0 4px 20px rgba(0,0,0,.2);
}
.accuracy-sub { opacity: .75; font-size: .92rem; margin-top: 4px; }
.accuracy-pills { display: flex; gap: 12px; justify-content: center; margin-top: 16px; flex-wrap: wrap; }
.pill-red, .pill-green {
  display: inline-flex; align-items: center;
  font-size: .85rem; font-weight: 600;
  padding: 5px 16px; border-radius: 100px;
}
.pill-red   { background: rgba(231,76,60,.25); border: 1px solid rgba(231,76,60,.5); }
.pill-green { background: rgba(22,163,74,.25); border: 1px solid rgba(22,163,74,.5); }

/* Result Cards */
.result-card {
  background: var(--card-bg);
  border-radius: var(--radius);
  box-shadow: var(--shadow-md);
  overflow: hidden;
}
.result-card-header {
  background: linear-gradient(90deg, var(--blue-dark), var(--blue-mid));
  color: #fff;
  font-family: 'Syne', sans-serif;
  font-size: .9rem;
  font-weight: 700;
  letter-spacing: 1.5px;
  text-transform: uppercase;
  padding: 14px 24px;
}
.result-card-body { padding: 24px; }

/* Report Box */
.report-box {
  background: #0f172a;
  color: #a5f3fc;
  font-family: 'Courier New', monospace;
  font-size: .82rem;
  line-height: 1.7;
  padding: 20px 24px;
  border-radius: 10px;
  overflow-x: auto;
  white-space: pre;
  border: 1px solid #1e3a5f;
  box-shadow: inset 0 2px 8px rgba(0,0,0,.3);
}

/* Chart */
.chart-card .result-card-body { padding: 16px; }
.chart-img {
  width: 100%;
  max-width: 420px;
  height: auto;
  border-radius: 10px;
  box-shadow: 0 2px 12px rgba(0,0,0,.08);
}

/* Download Section */
.download-card {}
.download-desc { color: var(--text-mid); font-size: .95rem; margin-bottom: 16px; }
.download-btn-row { display: flex; gap: 14px; flex-wrap: wrap; }
.btn-download {
  display: inline-flex;
  align-items: center;
  gap: 8px;
  padding: 13px 22px;
  border-radius: 50px;
  font-family: 'Syne', sans-serif;
  font-size: .9rem;
  font-weight: 700;
  text-decoration: none;
  transition: transform .15s, box-shadow .15s;
  box-shadow: 0 4px 16px rgba(0,0,0,.12);
}
.btn-download:hover { transform: translateY(-2px); box-shadow: 0 8px 24px rgba(0,0,0,.18); }
.btn-download-red   { background: linear-gradient(135deg, #b91c1c, #ef4444); color: #fff; }
.btn-download-green { background: linear-gradient(135deg, #15803d, #22c55e); color: #fff; }
.btn-count {
  font-size: .75rem;
  font-weight: 500;
  background: rgba(255,255,255,.2);
  padding: 2px 8px;
  border-radius: 100px;
  margin-left: 4px;
}

/* Upload Another Button */
.btn-upload-another {
  display: inline-flex;
  align-items: center;
  gap: 8px;
  background: #fff;
  color: var(--blue-mid);
  font-family: 'Syne', sans-serif;
  font-weight: 700;
  font-size: 1rem;
  padding: 14px 36px;
  border-radius: 50px;
  text-decoration: none;
  border: 2px solid var(--blue-mid);
  transition: background .15s, color .15s, transform .15s, box-shadow .15s;
  box-shadow: var(--shadow-sm);
}
.btn-upload-another:hover {
  background: var(--blue-mid);
  color: #fff;
  transform: translateY(-2px);
  box-shadow: 0 8px 24px rgba(30,92,179,.25);
}

/* Footer */
.site-footer-bottom {
  text-align: center;
  padding: 20px;
  color: var(--text-muted);
  font-size: .83rem;
  border-top: 1px solid #e2e8f0;
  background: #fff;
}

/* Responsive */
@media (max-width: 576px) {
  .card-body-inner { padding: 24px 20px; }
  .accuracy-banner { padding: 28px 20px; }
  .btn-upload { padding: 13px 28px; font-size: 1rem; }
  .download-btn-row { flex-direction: column; }
}

In [ ]:
# ============================================================
#  DDoS Detection System — Google Colab Setup & Run Guide
# ============================================================
#
#  STEP-BY-STEP INSTRUCTIONS
#  Copy each cell into a Google Colab notebook and run in order.
# ============================================================

# ── CELL 1: Install dependencies ────────────────────────────
"""
!pip install flask pyngrok scikit-learn pandas numpy matplotlib werkzeug
"""

# ── CELL 2: Create project folder structure ─────────────────
"""
import os

folders = [
    'ddos_app',
    'ddos_app/uploads',
    'ddos_app/static',
    'ddos_app/templates',
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("✅ Folder structure created")
"""

# ── CELL 3: Write app.py ─────────────────────────────────────
# (Copy the full app.py content from the provided app.py file into ddos_app/app.py)
"""
# Use %%writefile magic in Colab:
# %%writefile ddos_app/app.py
# <paste full app.py content here>
"""

# ── CELL 4: Write templates/index.html ──────────────────────
"""
# %%writefile ddos_app/templates/index.html
# <paste full index.html content here>
"""

# ── CELL 5: Write templates/result.html ─────────────────────
"""
# %%writefile ddos_app/templates/result.html
# <paste full result.html content here>
"""

# ── CELL 6: Write static/style.css ──────────────────────────
"""
# %%writefile ddos_app/static/style.css
# <paste full style.css content here>
"""

# ── CELL 7: Train & save a demo Random Forest model ─────────
"""
import pickle, numpy as np
from sklearn.ensemble import RandomForestClassifier

# Generates a simple model compatible with CIC-IDS2017 feature count.
# Replace this with your real trained model.
np.random.seed(42)
n_features = 78   # CIC-IDS2017 has 78 features after dropping Label
X_dummy = np.random.rand(200, n_features)
y_dummy = np.random.choice(['BENIGN', 'DDoS'], 200)

clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_dummy, y_dummy)

with open('ddos_app/model.pkl', 'wb') as f:
    pickle.dump(clf, f)

print("✅ model.pkl saved (demo model — replace with your trained model)")
"""

# ── CELL 8: Start Flask + ngrok tunnel ──────────────────────
"""
import os, threading
from pyngrok import ngrok

os.chdir('ddos_app')

# Set your ngrok authtoken (get free token at https://dashboard.ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

# Start Flask in a background thread
def run_flask():
    os.system('python app.py')

t = threading.Thread(target=run_flask)
t.daemon = True
t.start()

import time; time.sleep(2)  # Wait for Flask to start

# Open ngrok tunnel on port 5000
public_url = ngrok.connect(5000)
print(f"\\n🚀 DDoS Detection App is LIVE at: {public_url}")
print("   Open the URL above in your browser to use the app.")
"""

# ── NOTES ───────────────────────────────────────────────────
"""
NOTES:
- Upload a CSV file with network flow features (e.g. CIC-IDS2017 format).
- If your CSV has a 'Label' column with BENIGN/DDoS values, accuracy and
  classification report will be computed automatically.
- The app saves ddos_predictions.csv and benign_predictions.csv in the
  ddos_app/ directory.
- Charts are saved to ddos_app/static/ and rendered on the results page.
- To use your own trained model, replace ddos_app/model.pkl with your
  pickle-serialized sklearn model. The model must accept the same numeric
  feature columns as your training data (Label column is excluded).
"""